# Prediksi Biaya Asuransi Kesehatan — Data Cleaning & Feature Engineering

Notebook ini mempersiapkan data untuk pemodelan:
1. Menghapus duplikat
2. Encoding fitur kategorikal
3. Transformasi target (Log)
4. Menyimpan dataset bersih

## 1. Import & Load Data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Data/insurance.csv')
print(f'Shape awal: {df.shape}')
df.head()

Shape awal: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Menghapus Duplikat

In [2]:
print(f'Jumlah duplikat: {df.duplicated().sum()}')
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape setelah hapus duplikat: {df.shape}')

Jumlah duplikat: 1
Shape setelah hapus duplikat: (1337, 7)


## 3. Cek & Tangani Missing Values

In [3]:
print('Missing values per kolom:')
print(df.isnull().sum())
print(f'\nTotal: {df.isnull().sum().sum()} (Tidak ada missing values)')

Missing values per kolom:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Total: 0 (Tidak ada missing values)


## 4. Encoding Fitur Kategorikal

Mengubah kolom teks menjadi angka agar bisa diproses oleh model:
- `sex`: male=1, female=0
- `smoker`: yes=1, no=0
- `region`: One-Hot Encoding (membuat kolom baru untuk setiap wilayah)

In [4]:
df['sex'] = df['sex'].map({'male': 1, 'female': 0})
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})

df = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)

df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


## 5. Feature Engineering

Menambahkan fitur interaksi yang berpotensi meningkatkan akurasi model:
- `age_squared`: Umur kuadrat (biaya asuransi naik eksponensial seiring umur)
- `bmi_smoker`: Interaksi BMI × Smoker (BMI tinggi + perokok = biaya sangat mahal)

In [5]:
df['age_squared'] = df['age'] ** 2
df['bmi_smoker'] = df['bmi'] * df['smoker']

print(f'Shape setelah feature engineering: {df.shape}')
df.head()

Shape setelah feature engineering: (1337, 11)


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest,age_squared,bmi_smoker
0,19,0,27.900,0,1,16884.92400,0,0,1,361,27.9
1,18,1,33.770,1,0,1725.55230,0,1,0,324,0.0
2,28,1,33.000,3,0,4449.46200,0,1,0,784,0.0
3,33,1,22.705,0,0,21984.47061,1,0,0,1089,0.0
4,32,1,28.880,0,0,3866.85520,1,0,0,1024,0.0


## 6. Simpan Dataset Bersih

In [6]:
output_path = 'Data/insurance_clean.csv'
df.to_csv(output_path, index=False)

print(f'Dataset bersih tersimpan di: {output_path}')
print(f'Shape akhir: {df.shape}')
print(f'\nKolom-kolom:')
for col in df.columns:
    print(f'  - {col}')

Dataset bersih tersimpan di: Data/insurance_clean.csv
Shape akhir: (1337, 11)

Kolom-kolom:
  - age
  - sex
  - bmi
  - children
  - smoker
  - charges
  - region_northwest
  - region_southeast
  - region_southwest
  - age_squared
  - bmi_smoker
